In [0]:
%sql
CREATE WIDGET TEXT catalog DEFAULT "bootcamp";
CREATE WIDGET TEXT bronze_schema DEFAULT "urbanmart_bronze_panagiotakopoulos_026";
CREATE WIDGET TEXT silver_schema DEFAULT "urbanmart_silver_panagiotakopoulos_026";
CREATE WIDGET TEXT gold_schema DEFAULT "urbanmart_gold_panagiotakopoulos_026";
CREATE WIDGET TEXT source_path DEFAULT "/Volumes/bootcamp/default/volume";

CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :bronze_schema) COMMENT 'Urbanmart star schema (bronze / silver / gold).';
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :silver_schema) COMMENT 'Urbanmart star schema (bronze / silver / gold).';
CREATE SCHEMA IF NOT EXISTS IDENTIFIER(:catalog || '.' || :gold_schema) COMMENT 'Urbanmart star schema (bronze / silver / gold).';

USE CATALOG IDENTIFIER(:catalog);
USE SCHEMA IDENTIFIER(:bronze_schema);

In [0]:
%sql

-- 1. CUSTOMERS
CREATE TABLE IF NOT EXISTS customers (
    customer_id STRING COMMENT 'Raw customer identifier',
    first_name STRING COMMENT 'Raw first name',
    last_name STRING COMMENT 'Raw last name',
    email STRING COMMENT 'Raw email',
    gender STRING COMMENT 'Raw gender',
    birth_date STRING COMMENT 'Raw birth date',
    city STRING COMMENT 'Raw city',
    region STRING COMMENT 'Raw region',
    segment STRING COMMENT 'Raw customer segment',
    signup_date STRING COMMENT 'Raw signup date',
    _source_file  STRING COMMENT 'Source file path',
    _file_version STRING COMMENT 'Fingerprint of the source file version',
    _batch_id STRING COMMENT 'Load run that wrote this row',
    _ingested_at TIMESTAMP COMMENT 'Ingestion timestamp'
)
USING DELTA
COMMENT 'Bronze raw customer data';

-- 2. CUSTOMER CHANGES
CREATE TABLE IF NOT EXISTS customer_changes (
    customer_id STRING COMMENT 'Raw customer identifier',
    new_city STRING COMMENT 'New city',
    new_region STRING COMMENT 'New region',
    new_segment STRING COMMENT 'New customer segment',
    effective_date STRING COMMENT 'Raw effective date',
    _source_file  STRING COMMENT 'Source file path',
    _file_version STRING COMMENT 'Fingerprint of the source file version',
    _batch_id     STRING COMMENT 'Load run that wrote this row',
    _ingested_at  TIMESTAMP COMMENT 'Ingestion timestamp'
)
USING DELTA
COMMENT 'Bronze raw customer change events';

-- 3. PRODUCTS
CREATE TABLE IF NOT EXISTS products (
    product_id STRING COMMENT 'Raw product identifier',
    product_name STRING COMMENT 'Raw product name',
    category STRING COMMENT 'Raw category',
    subcategory STRING COMMENT 'Raw subcategory',
    brand STRING COMMENT 'Raw brand',
    unit_cost STRING COMMENT 'Raw unit cost',
    list_price STRING COMMENT 'Raw list price',
    is_active STRING COMMENT 'Raw active flag',
    _source_file  STRING COMMENT 'Source file path',
    _file_version STRING COMMENT 'Fingerprint of the source file version',
    _batch_id     STRING COMMENT 'Load run that wrote this row',
    _ingested_at  TIMESTAMP COMMENT 'Ingestion timestamp'
)
USING DELTA
COMMENT 'Bronze raw product catalogue';

-- 4. STORES
CREATE TABLE IF NOT EXISTS stores (
    store_id STRING COMMENT 'Raw store identifier',
    store_name STRING COMMENT 'Raw store name',
    store_type STRING COMMENT 'Raw store type',
    city STRING COMMENT 'Raw city',
    region STRING COMMENT 'Raw region',
    open_date STRING COMMENT 'Raw opening date',
    size_sqm STRING COMMENT 'Raw store size in square metres',
    _source_file STRING COMMENT 'Source file path',
    _file_version STRING COMMENT 'Fingerprint of the source file version',
    _batch_id STRING COMMENT 'Load run that wrote this row',
    _ingested_at TIMESTAMP COMMENT 'Ingestion timestamp'
)
USING DELTA
COMMENT 'Bronze raw store data';

-- 5. ORDERS
CREATE TABLE IF NOT EXISTS orders (
    order_id STRING COMMENT 'Raw order identifier',
    order_ts STRING COMMENT 'Raw order timestamp',
    customer_id STRING COMMENT 'Raw customer identifier',
    store_id STRING COMMENT 'Raw store identifier',
    payment_method STRING COMMENT 'Raw payment method',
    order_status STRING COMMENT 'Raw order status',
    _source_file STRING COMMENT 'Source file path',
    _file_version STRING COMMENT 'Fingerprint of the source file version',
    _batch_id STRING COMMENT 'Load run that wrote this row',
    _ingested_at TIMESTAMP COMMENT 'Ingestion timestamp'
)
USING DELTA
COMMENT 'Bronze raw order headers';

-- 6. ORDER ITEMS
CREATE TABLE IF NOT EXISTS order_items (
    order_line_id STRING COMMENT 'Raw order line identifier',
    order_id STRING COMMENT 'Raw order identifier',
    product_id STRING COMMENT 'Raw product identifier',
    quantity STRING COMMENT 'Raw quantity',
    unit_price STRING COMMENT 'Raw unit price',
    discount_pct STRING COMMENT 'Raw discount percentage',
    _source_file STRING COMMENT 'Source file path',
    _file_version STRING COMMENT 'Fingerprint of the source file version',
    _batch_id STRING COMMENT 'Load run that wrote this row',
    _ingested_at TIMESTAMP COMMENT 'Ingestion timestamp'
)
USING DELTA
COMMENT 'Bronze raw order line data';

-- 7. DATA QUALITY LOG
CREATE TABLE IF NOT EXISTS dq_log (
    id BIGINT GENERATED ALWAYS AS IDENTITY COMMENT 'Unique identifier for each data quality finding',
    source_table STRING NOT NULL COMMENT 'Name of the Bronze source table being profiled',
    column_name STRING COMMENT 'Column associated with the data quality issue, or ALL_COLUMNS for record-level issues',
    issue_description STRING NOT NULL COMMENT 'Description of the detected data quality issue',
    row_affected BIGINT COMMENT 'Number of rows affected by the detected issue',
    handling_rule STRING NOT NULL COMMENT 'Recommended handling action, such as flagging, deduplication, or rejection from Silver',
    profiled_at TIMESTAMP COMMENT 'Timestamp when the data quality check was performed'
)
USING DELTA
COMMENT 'Records data quality profiling findings, affected row counts, and recommended handling rules for Bronze data before processing into Silver.';

CREATE TABLE IF NOT EXISTS bronze_file_ledger (
    source_file STRING NOT NULL COMMENT 'Full path of the ingested source file',
    file_version STRING NOT NULL COMMENT 'Fingerprint identifying a specific version of the source file',
    source_table STRING NOT NULL COMMENT 'Target Bronze table associated with the source file',
    status STRING NOT NULL COMMENT 'Ingestion status, such as SUCCESS or FAILED',
    row_count BIGINT COMMENT 'Number of source rows processed for this file version',
    ingested_at TIMESTAMP COMMENT 'Timestamp when the ingestion status was recorded',
    batch_id STRING COMMENT 'Identifier linking the file ingestion to a processing batch'
)
USING DELTA
COMMENT 'Tracks Bronze ingestion status by source file and file version to support incremental ingestion, auditability, and recovery.';

In [0]:
%sql

DECLARE OR REPLACE VARIABLE v_batch_id STRING;
SET VAR v_batch_id = uuid();

-- 1. CUSTOMERS
INSERT INTO customers
REPLACE WHERE endswith(_source_file, '/customers.csv')
SELECT
    customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date,
    _metadata.file_path AS _source_file,
    sha2(concat_ws('|', _metadata.file_path, CAST(_metadata.file_size AS STRING), CAST(_metadata.file_modification_time AS STRING)), 256) AS _file_version,
    v_batch_id AS _batch_id,
    current_timestamp() AS _ingested_at
FROM read_files(:source_path || '/customers.csv', format => 'csv', header => true, inferColumnTypes => false);

INSERT INTO bronze_file_ledger (source_file, file_version, source_table, status, row_count, ingested_at, batch_id)
SELECT _source_file, _file_version, 'customers', 'SUCCESS', COUNT(*), current_timestamp(), v_batch_id
FROM customers
WHERE _batch_id = v_batch_id
GROUP BY _source_file, _file_version;

-- 2. CUSTOMER CHANGES
INSERT INTO customer_changes
REPLACE WHERE endswith(_source_file, '/customer_changes.csv')
SELECT
    customer_id, new_city, new_region, new_segment, effective_date,
    _metadata.file_path AS _source_file,
    sha2(concat_ws('|', _metadata.file_path, CAST(_metadata.file_size AS STRING), CAST(_metadata.file_modification_time AS STRING)), 256) AS _file_version,
    v_batch_id AS _batch_id,
    current_timestamp() AS _ingested_at
FROM read_files(:source_path || '/customer_changes.csv', format => 'csv', header => true, inferColumnTypes => false);

INSERT INTO bronze_file_ledger (source_file, file_version, source_table, status, row_count, ingested_at, batch_id)
SELECT _source_file, _file_version, 'customer_changes', 'SUCCESS', COUNT(*), current_timestamp(), v_batch_id
FROM customer_changes
WHERE _batch_id = v_batch_id
GROUP BY _source_file, _file_version;

-- 3. PRODUCTS
INSERT INTO products
REPLACE WHERE endswith(_source_file, '/products.csv')
SELECT
    product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active,
    _metadata.file_path AS _source_file,
    sha2(concat_ws('|', _metadata.file_path, CAST(_metadata.file_size AS STRING), CAST(_metadata.file_modification_time AS STRING)), 256) AS _file_version,
    v_batch_id AS _batch_id,
    current_timestamp() AS _ingested_at
FROM read_files(:source_path || '/products.csv', format => 'csv', header => true, inferColumnTypes => false);

INSERT INTO bronze_file_ledger (source_file, file_version, source_table, status, row_count, ingested_at, batch_id)
SELECT _source_file, _file_version, 'products', 'SUCCESS', COUNT(*), current_timestamp(), v_batch_id
FROM products
WHERE _batch_id = v_batch_id
GROUP BY _source_file, _file_version;

-- 4. STORES
INSERT INTO stores
REPLACE WHERE endswith(_source_file, '/stores.csv')
SELECT
    store_id, store_name, store_type, city, region, open_date, size_sqm,
    _metadata.file_path AS _source_file,
    sha2(concat_ws('|', _metadata.file_path, CAST(_metadata.file_size AS STRING), CAST(_metadata.file_modification_time AS STRING)), 256) AS _file_version,
    v_batch_id AS _batch_id,
    current_timestamp() AS _ingested_at
FROM read_files(:source_path || '/stores.csv', format => 'csv', header => true, inferColumnTypes => false);

INSERT INTO bronze_file_ledger (source_file, file_version, source_table, status, row_count, ingested_at, batch_id)
SELECT _source_file, _file_version, 'stores', 'SUCCESS', COUNT(*), current_timestamp(), v_batch_id
FROM stores
WHERE _batch_id = v_batch_id
GROUP BY _source_file, _file_version;

-- 5. ORDERS
INSERT INTO orders
REPLACE WHERE endswith(_source_file, '/orders.csv')
SELECT
    order_id, order_ts, customer_id, store_id, payment_method, order_status,
    _metadata.file_path AS _source_file,
    sha2(concat_ws('|', _metadata.file_path, CAST(_metadata.file_size AS STRING), CAST(_metadata.file_modification_time AS STRING)), 256) AS _file_version,
    v_batch_id AS _batch_id,
    current_timestamp() AS _ingested_at
FROM read_files(:source_path || '/orders.csv', format => 'csv', header => true, inferColumnTypes => false);

INSERT INTO bronze_file_ledger (source_file, file_version, source_table, status, row_count, ingested_at, batch_id)
SELECT _source_file, _file_version, 'orders', 'SUCCESS', COUNT(*), current_timestamp(), v_batch_id
FROM orders
WHERE _batch_id = v_batch_id
GROUP BY _source_file, _file_version;

-- 6. ORDER ITEMS
INSERT INTO order_items
REPLACE WHERE endswith(_source_file, '/order_items.csv')
SELECT
    order_line_id, order_id, product_id, quantity, unit_price, discount_pct,
    _metadata.file_path AS _source_file,
    sha2(concat_ws('|', _metadata.file_path, CAST(_metadata.file_size AS STRING), CAST(_metadata.file_modification_time AS STRING)), 256) AS _file_version,
    v_batch_id AS _batch_id,
    current_timestamp() AS _ingested_at
FROM read_files(:source_path || '/order_items.csv', format => 'csv', header => true, inferColumnTypes => false);

INSERT INTO bronze_file_ledger (source_file, file_version, source_table, status, row_count, ingested_at, batch_id)
SELECT _source_file, _file_version, 'order_items', 'SUCCESS', COUNT(*), current_timestamp(), v_batch_id
FROM order_items
WHERE _batch_id = v_batch_id
GROUP BY _source_file, _file_version;

num_affected_rows,num_inserted_rows
1,1


In [0]:
%sql

WITH src AS (
    SELECT 'customers' AS t, COUNT(*) AS src_rows, COUNT_IF(_rescued_data IS NOT NULL) AS rescued_rows
    FROM read_files(:source_path || '/customers.csv', format => 'csv', header => true, inferColumnTypes => false)
    UNION ALL
    SELECT 'customer_changes', COUNT(*), COUNT_IF(_rescued_data IS NOT NULL)
    FROM read_files(:source_path || '/customer_changes.csv', format => 'csv', header => true, inferColumnTypes => false)
    UNION ALL
    SELECT 'products', COUNT(*), COUNT_IF(_rescued_data IS NOT NULL)
    FROM read_files(:source_path || '/products.csv', format => 'csv', header => true, inferColumnTypes => false)
    UNION ALL
    SELECT 'stores', COUNT(*), COUNT_IF(_rescued_data IS NOT NULL)
    FROM read_files(:source_path || '/stores.csv', format => 'csv', header => true, inferColumnTypes => false)
    UNION ALL
    SELECT 'orders', COUNT(*), COUNT_IF(_rescued_data IS NOT NULL)
    FROM read_files(:source_path || '/orders.csv', format => 'csv', header => true, inferColumnTypes => false)
    UNION ALL
    SELECT 'order_items', COUNT(*), COUNT_IF(_rescued_data IS NOT NULL)
    FROM read_files(:source_path || '/order_items.csv', format => 'csv', header => true, inferColumnTypes => false)
),
brz AS (
    SELECT 'customers' AS t,
           COUNT(*) AS brz_rows,
           COUNT(DISTINCT _file_version) AS versions,
           COUNT(DISTINCT _batch_id) AS batches,
           COUNT_IF(_source_file IS NULL OR _file_version IS NULL OR _batch_id IS NULL OR _ingested_at IS NULL) AS missing_lineage
    FROM customers
    UNION ALL
    SELECT 'customer_changes', COUNT(*), COUNT(DISTINCT _file_version), COUNT(DISTINCT _batch_id),
           COUNT_IF(_source_file IS NULL OR _file_version IS NULL OR _batch_id IS NULL OR _ingested_at IS NULL)
    FROM customer_changes
    UNION ALL
    SELECT 'products', COUNT(*), COUNT(DISTINCT _file_version), COUNT(DISTINCT _batch_id),
           COUNT_IF(_source_file IS NULL OR _file_version IS NULL OR _batch_id IS NULL OR _ingested_at IS NULL)
    FROM products
    UNION ALL
    SELECT 'stores', COUNT(*), COUNT(DISTINCT _file_version), COUNT(DISTINCT _batch_id),
           COUNT_IF(_source_file IS NULL OR _file_version IS NULL OR _batch_id IS NULL OR _ingested_at IS NULL)
    FROM stores
    UNION ALL
    SELECT 'orders', COUNT(*), COUNT(DISTINCT _file_version), COUNT(DISTINCT _batch_id),
           COUNT_IF(_source_file IS NULL OR _file_version IS NULL OR _batch_id IS NULL OR _ingested_at IS NULL)
    FROM orders
    UNION ALL
    SELECT 'order_items', COUNT(*), COUNT(DISTINCT _file_version), COUNT(DISTINCT _batch_id),
           COUNT_IF(_source_file IS NULL OR _file_version IS NULL OR _batch_id IS NULL OR _ingested_at IS NULL)
    FROM order_items
),
led AS (
    -- Latest successful ledger entry per table
    SELECT source_table AS t, row_count AS ledger_rows
    FROM bronze_file_ledger
    WHERE status = 'SUCCESS'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY source_table ORDER BY ingested_at DESC) = 1
)
SELECT
    s.t AS table_name,
    s.src_rows,
    b.brz_rows,
    l.ledger_rows,
    assert_true(s.src_rows > 0,
        format_string('Source file is empty: %s', s.t)) AS source_not_empty,
    assert_true(s.rescued_rows = 0,
        format_string('Source has %d rows with rescued (unparsed) data: %s', s.rescued_rows, s.t)) AS no_rescued_data,
    assert_true(s.src_rows = b.brz_rows,
        format_string('Bronze row count differs from source: %s (source %d, bronze %d)', s.t, s.src_rows, b.brz_rows)) AS counts_match,
    assert_true(b.versions = 1 AND b.batches = 1,
        format_string('Bronze holds more than one file version or batch: %s (versions %d, batches %d)', s.t, b.versions, b.batches)) AS single_version_and_batch,
    assert_true(b.missing_lineage = 0,
        format_string('Bronze rows with NULL lineage columns: %s (%d rows)', s.t, b.missing_lineage)) AS lineage_complete,
    assert_true(COALESCE(l.ledger_rows, -1) = s.src_rows,
        format_string('Latest ledger row count differs from source: %s (source %d, ledger %d; -1 = no ledger row)', s.t, s.src_rows, COALESCE(l.ledger_rows, -1))) AS ledger_matches
FROM src s
JOIN brz b ON s.t = b.t
LEFT JOIN led l ON s.t = l.t
ORDER BY s.t;

table_name,src_rows,brz_rows,ledger_rows,source_not_empty,no_rescued_data,counts_match,single_version_and_batch,lineage_complete,ledger_matches
customer_changes,25,25,25,null,null,null,null,null,null
customers,508,508,508,null,null,null,null,null,null
order_items,24111,24111,24111,null,null,null,null,null,null
orders,8000,8000,8000,null,null,null,null,null,null
products,76,76,76,null,null,null,null,null,null
stores,13,13,13,null,null,null,null,null,null


In [0]:
%sql

TRUNCATE TABLE dq_log;
 
 -- 1. CHECK FOR EXACT DUPLICATE ENTRIES (row_affected = extra rows beyond the first)
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
SELECT
    'customers',
    'ALL_COLUMNS',
    'Exact duplicate customer records',
    COALESCE(SUM(duplicate_count - 1), 0),
    'DEDUPLICATE: remove exact duplicates before Silver',
    CURRENT_TIMESTAMP()
FROM (
    SELECT COUNT(*) AS duplicate_count
    FROM customers
    GROUP BY customer_id, first_name, last_name, email, gender, birth_date, city, region, segment, signup_date
    HAVING COUNT(*) > 1
) d
UNION ALL
SELECT
    'customer_changes',
    'ALL_COLUMNS',
    'Exact duplicate customer change records',
    COALESCE(SUM(duplicate_count - 1), 0),
    'DEDUPLICATE: remove exact duplicates before Silver',
    CURRENT_TIMESTAMP()
FROM (
    SELECT COUNT(*) AS duplicate_count
    FROM customer_changes
    GROUP BY customer_id, new_city, new_region, new_segment, effective_date
    HAVING COUNT(*) > 1
) d
UNION ALL
SELECT
    'products',
    'ALL_COLUMNS',
    'Exact duplicate product records',
    COALESCE(SUM(duplicate_count - 1), 0),
    'DEDUPLICATE: remove exact duplicates before Silver',
    CURRENT_TIMESTAMP()
FROM (
    SELECT COUNT(*) AS duplicate_count
    FROM products
    GROUP BY product_id, product_name, category, subcategory, brand, unit_cost, list_price, is_active
    HAVING COUNT(*) > 1
) d
UNION ALL
SELECT
    'stores',
    'ALL_COLUMNS',
    'Exact duplicate store records',
    COALESCE(SUM(duplicate_count - 1), 0),
    'DEDUPLICATE: remove exact duplicates before Silver',
    CURRENT_TIMESTAMP()
FROM (
    SELECT COUNT(*) AS duplicate_count
    FROM stores
    GROUP BY store_id, store_name, store_type, city, region, open_date, size_sqm
    HAVING COUNT(*) > 1
) d
UNION ALL
SELECT
    'orders',
    'ALL_COLUMNS',
    'Exact duplicate order records',
    COALESCE(SUM(duplicate_count - 1), 0),
    'DEDUPLICATE: remove exact duplicates before Silver',
    CURRENT_TIMESTAMP()
FROM (
    SELECT COUNT(*) AS duplicate_count
    FROM orders
    GROUP BY order_id, order_ts, customer_id, store_id, payment_method, order_status
    HAVING COUNT(*) > 1
) d
UNION ALL
SELECT
    'order_items',
    'ALL_COLUMNS',
    'Exact duplicate order item records',
    COALESCE(SUM(duplicate_count - 1), 0),
    'DEDUPLICATE: remove exact duplicates before Silver',
    CURRENT_TIMESTAMP()
FROM (
    SELECT COUNT(*) AS duplicate_count
    FROM order_items
    GROUP BY order_line_id, order_id, product_id, quantity, unit_price, discount_pct
    HAVING COUNT(*) > 1
) d;
 
-- 2. PROFILE NULL AND BLANK VALUES: DEFAULT TO UNKNOWN
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
WITH missing_values AS (
    SELECT 'customers' AS table_name, 'first_name' AS column_name,
        COUNT_IF(NULLIF(TRIM(first_name), '') IS NULL) AS missing_count
    FROM customers
    UNION ALL
    SELECT 'customers', 'last_name',
        COUNT_IF(NULLIF(TRIM(last_name), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'email',
        COUNT_IF(NULLIF(TRIM(email), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'gender',
        COUNT_IF(NULLIF(TRIM(gender), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'birth_date',
        COUNT_IF(NULLIF(TRIM(birth_date), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'city',
        COUNT_IF(NULLIF(TRIM(city), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'region',
        COUNT_IF(NULLIF(TRIM(region), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'segment',
        COUNT_IF(NULLIF(TRIM(segment), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customers', 'signup_date',
        COUNT_IF(NULLIF(TRIM(signup_date), '') IS NULL)
    FROM customers
    UNION ALL
    SELECT 'customer_changes', 'new_city',
        COUNT_IF(NULLIF(TRIM(new_city), '') IS NULL)
    FROM customer_changes
    UNION ALL
    SELECT 'customer_changes', 'new_region',
        COUNT_IF(NULLIF(TRIM(new_region), '') IS NULL)
    FROM customer_changes
    UNION ALL
    SELECT 'customer_changes', 'new_segment',
        COUNT_IF(NULLIF(TRIM(new_segment), '') IS NULL)
    FROM customer_changes
    UNION ALL
    SELECT 'products', 'subcategory',
        COUNT_IF(NULLIF(TRIM(subcategory), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'products', 'brand',
        COUNT_IF(NULLIF(TRIM(brand), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'products', 'is_active',
        COUNT_IF(NULLIF(TRIM(is_active), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'stores', 'store_name',
        COUNT_IF(NULLIF(TRIM(store_name), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'stores', 'store_type',
        COUNT_IF(NULLIF(TRIM(store_type), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'stores', 'city',
        COUNT_IF(NULLIF(TRIM(city), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'stores', 'region',
        COUNT_IF(NULLIF(TRIM(region), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'stores', 'open_date',
        COUNT_IF(NULLIF(TRIM(open_date), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'stores', 'size_sqm',
        COUNT_IF(NULLIF(TRIM(size_sqm), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'orders', 'order_ts',
        COUNT_IF(NULLIF(TRIM(order_ts), '') IS NULL)
    FROM orders
    UNION ALL
    SELECT 'orders', 'customer_id',
        COUNT_IF(NULLIF(TRIM(customer_id), '') IS NULL)
    FROM orders
    UNION ALL
    SELECT 'orders', 'store_id',
        COUNT_IF(NULLIF(TRIM(store_id), '') IS NULL)
    FROM orders
    UNION ALL
    SELECT 'orders', 'payment_method',
        COUNT_IF(NULLIF(TRIM(payment_method), '') IS NULL)
    FROM orders
    UNION ALL
    SELECT 'order_items', 'discount_pct',
        COUNT_IF(NULLIF(TRIM(discount_pct), '') IS NULL)
    FROM order_items
)
SELECT
    table_name,
    column_name,
    CONCAT('Missing or blank values found in column ', column_name),
    missing_count,
    CASE
        WHEN table_name = 'order_items' AND column_name = 'discount_pct'
            THEN 'DEFAULT: treat missing discount as 0'
        WHEN table_name = 'orders' AND column_name IN ('customer_id', 'store_id')
            THEN 'DEFAULT: map to unknown member (-1)'
        ELSE 'DEFAULT: set to Unknown'
    END,
    CURRENT_TIMESTAMP()
FROM missing_values;
 
-- 3. PROFILE NULL AND BLANK VALUES: REJECT FROM SILVER
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
WITH missing_values AS (
    SELECT 'customers' AS table_name, 'customer_id' AS column_name,
        COUNT_IF(NULLIF(TRIM(customer_id), '') IS NULL) AS missing_count
    FROM customers
    UNION ALL
    SELECT 'customer_changes', 'customer_id',
        COUNT_IF(NULLIF(TRIM(customer_id), '') IS NULL)
    FROM customer_changes
    UNION ALL
    SELECT 'customer_changes', 'effective_date',
        COUNT_IF(NULLIF(TRIM(effective_date), '') IS NULL)
    FROM customer_changes
    UNION ALL
    SELECT 'products', 'product_id',
        COUNT_IF(NULLIF(TRIM(product_id), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'products', 'product_name',
        COUNT_IF(NULLIF(TRIM(product_name), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'products', 'category',
        COUNT_IF(NULLIF(TRIM(category), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'products', 'unit_cost',
        COUNT_IF(NULLIF(TRIM(unit_cost), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'products', 'list_price',
        COUNT_IF(NULLIF(TRIM(list_price), '') IS NULL)
    FROM products
    UNION ALL
    SELECT 'stores', 'store_id',
        COUNT_IF(NULLIF(TRIM(store_id), '') IS NULL)
    FROM stores
    UNION ALL
    SELECT 'orders', 'order_id',
        COUNT_IF(NULLIF(TRIM(order_id), '') IS NULL)
    FROM orders
    UNION ALL
    SELECT 'orders', 'order_status',
        COUNT_IF(NULLIF(TRIM(order_status), '') IS NULL)
    FROM orders
    UNION ALL
    SELECT 'order_items', 'order_line_id',
        COUNT_IF(NULLIF(TRIM(order_line_id), '') IS NULL)
    FROM order_items
    UNION ALL
    SELECT 'order_items', 'order_id',
        COUNT_IF(NULLIF(TRIM(order_id), '') IS NULL)
    FROM order_items
    UNION ALL
    SELECT 'order_items', 'product_id',
        COUNT_IF(NULLIF(TRIM(product_id), '') IS NULL)
    FROM order_items
    UNION ALL
    SELECT 'order_items', 'quantity',
        COUNT_IF(NULLIF(TRIM(quantity), '') IS NULL)
    FROM order_items
    UNION ALL
    SELECT 'order_items', 'unit_price',
        COUNT_IF(NULLIF(TRIM(unit_price), '') IS NULL)
    FROM order_items
)
SELECT
    table_name,
    column_name,
    CONCAT('Missing or blank values found in column ', column_name),
    missing_count,
    'REJECT: missing required value',
    CURRENT_TIMESTAMP()
FROM missing_values;
 
-- 4. PROFILE INVALID DATA TYPES: REVIEW
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
WITH validation AS (
    SELECT
        'customers' AS table_name,
        'birth_date' AS column_name,
        'DATE (yyyy-MM-dd)' AS expected_type,
        COUNT_IF(NULLIF(TRIM(birth_date), '') IS NOT NULL AND TRY_TO_DATE(TRIM(birth_date), 'yyyy-MM-dd') IS NULL) AS invalid_rows
    FROM customers
    UNION ALL
    SELECT
        'customers', 'signup_date', 'DATE (yyyy-MM-dd)',
        COUNT_IF(NULLIF(TRIM(signup_date), '') IS NOT NULL AND TRY_TO_DATE(TRIM(signup_date), 'yyyy-MM-dd') IS NULL)
    FROM customers
    UNION ALL
    SELECT
        'stores', 'open_date', 'DATE (yyyy-MM-dd)',
        COUNT_IF(NULLIF(TRIM(open_date), '') IS NOT NULL AND TRY_TO_DATE(TRIM(open_date), 'yyyy-MM-dd') IS NULL)
    FROM stores
    UNION ALL
    SELECT
        'stores', 'size_sqm', 'INT',
        COUNT_IF(NULLIF(TRIM(size_sqm), '') IS NOT NULL AND TRY_CAST(TRIM(size_sqm) AS INT) IS NULL)
    FROM stores
    UNION ALL
    SELECT
        'orders', 'order_ts', 'TIMESTAMP (yyyy-MM-dd HH:mm:ss or dd/MM/yyyy HH:mm)',
        COUNT_IF(NULLIF(TRIM(order_ts), '') IS NOT NULL AND COALESCE(TRY_TO_TIMESTAMP(TRIM(order_ts), 'yyyy-MM-dd HH:mm:ss'), TRY_TO_TIMESTAMP(TRIM(order_ts), 'dd/MM/yyyy HH:mm')) IS NULL)
    FROM orders
    UNION ALL
    SELECT
        'order_items', 'discount_pct', 'DOUBLE',
        COUNT_IF(NULLIF(TRIM(discount_pct), '') IS NOT NULL AND TRY_CAST(TRIM(discount_pct) AS DOUBLE) IS NULL)
    FROM order_items
)
SELECT
    table_name,
    column_name,
    CONCAT('Invalid value: expected ', expected_type, ' type'),
    invalid_rows,
    'REVIEW: invalid data type value',
    CURRENT_TIMESTAMP()
FROM validation;
 
-- 5. PROFILE INVALID DATA TYPES: REJECT FROM SILVER
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
WITH validation AS (
    SELECT 'customer_changes' AS table_name, 'effective_date' AS column_name, 'DATE (yyyy-MM-dd)' AS expected_type,
        COUNT_IF(NULLIF(TRIM(effective_date), '') IS NOT NULL AND TRY_TO_DATE(TRIM(effective_date), 'yyyy-MM-dd') IS NULL) AS invalid_rows
    FROM customer_changes
    UNION ALL
    SELECT 'products', 'unit_cost', 'DECIMAL(12,2)',
        COUNT_IF(NULLIF(TRIM(unit_cost), '') IS NOT NULL AND (TRY_CAST(TRIM(unit_cost) AS DECIMAL(12,2)) IS NULL OR LENGTH(REGEXP_EXTRACT(TRIM(unit_cost), '\\.([0-9]+)$', 1)) > 2))
    FROM products
    UNION ALL
    SELECT 'products', 'list_price', 'DECIMAL(12,2)',
        COUNT_IF(NULLIF(TRIM(list_price), '') IS NOT NULL AND (TRY_CAST(TRIM(list_price) AS DECIMAL(12,2)) IS NULL OR LENGTH(REGEXP_EXTRACT(TRIM(list_price), '\\.([0-9]+)$', 1)) > 2))
    FROM products
    UNION ALL
    SELECT 'order_items', 'quantity', 'INT',
        COUNT_IF(NULLIF(TRIM(quantity), '') IS NOT NULL AND TRY_CAST(TRIM(quantity) AS INT) IS NULL)
    FROM order_items
    UNION ALL
    SELECT 'order_items', 'unit_price', 'DECIMAL(12,2)',
        COUNT_IF(NULLIF(TRIM(unit_price), '') IS NOT NULL AND TRY_CAST(TRIM(unit_price) AS DECIMAL(12,2)) IS NULL)
    FROM order_items
)
SELECT
    table_name,
    column_name,
    CONCAT('Invalid value: expected ', expected_type, ' type'),
    invalid_rows,
    'REJECT: invalid data type value',
    CURRENT_TIMESTAMP()
FROM validation;
 
-- 6. CHECK CONFLICTING DUPLICATE IDENTIFIERS
--    row_affected = ALL rows in the conflicting key groups (not just the extras).
--    to_json(struct(...)) treats NULL vs value as a conflict, unlike COUNT(DISTINCT col).
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
SELECT
    'customers',
    'customer_id',
    'Rows in a customer_id group with conflicting data',
    COUNT(*),
    'REVIEW: resolve conflicts before Silver',
    CURRENT_TIMESTAMP()
FROM customers
WHERE customer_id IS NOT NULL AND TRIM(customer_id) <> '' AND customer_id IN (
    SELECT customer_id
    FROM customers
    WHERE customer_id IS NOT NULL AND TRIM(customer_id) <> ''
    GROUP BY customer_id
    HAVING COUNT(DISTINCT to_json(struct(first_name, last_name, email, gender, birth_date, city, region, segment, signup_date))) > 1
)
UNION ALL
SELECT
    'products',
    'product_id',
    'Rows in a product_id group with conflicting data',
    COUNT(*),
    'REVIEW: resolve conflicts before Silver',
    CURRENT_TIMESTAMP()
FROM products
WHERE product_id IS NOT NULL AND TRIM(product_id) <> '' AND product_id IN (
    SELECT product_id
    FROM products
    WHERE product_id IS NOT NULL AND TRIM(product_id) <> ''
    GROUP BY product_id
    HAVING COUNT(DISTINCT to_json(struct(product_name, category, subcategory, brand, unit_cost, list_price, is_active))) > 1
)
UNION ALL
SELECT
    'stores',
    'store_id',
    'Rows in a store_id group with conflicting data',
    COUNT(*),
    'REVIEW: resolve conflicts before Silver',
    CURRENT_TIMESTAMP()
FROM stores
WHERE store_id IS NOT NULL AND TRIM(store_id) <> '' AND store_id IN (
    SELECT store_id
    FROM stores
    WHERE store_id IS NOT NULL AND TRIM(store_id) <> ''
    GROUP BY store_id
    HAVING COUNT(DISTINCT to_json(struct(store_name, store_type, city, region, open_date, size_sqm))) > 1
)
UNION ALL
SELECT
    'orders',
    'order_id',
    'Rows in an order_id group with conflicting data',
    COUNT(*),
    'REVIEW: resolve conflicts before Silver',
    CURRENT_TIMESTAMP()
FROM orders
WHERE order_id IS NOT NULL AND TRIM(order_id) <> '' AND order_id IN (
    SELECT order_id
    FROM orders
    WHERE order_id IS NOT NULL AND TRIM(order_id) <> ''
    GROUP BY order_id
    HAVING COUNT(DISTINCT to_json(struct(order_ts, customer_id, store_id, payment_method, order_status))) > 1
)
UNION ALL
SELECT
    'order_items',
    'order_line_id',
    'Rows in an order_line_id group with conflicting data',
    COUNT(*),
    'REVIEW: resolve conflicts before Silver',
    CURRENT_TIMESTAMP()
FROM order_items
WHERE order_line_id IS NOT NULL AND TRIM(order_line_id) <> '' AND order_line_id IN (
    SELECT order_line_id
    FROM order_items
    WHERE order_line_id IS NOT NULL AND TRIM(order_line_id) <> ''
    GROUP BY order_line_id
    HAVING COUNT(DISTINCT to_json(struct(order_id, product_id, quantity, unit_price, discount_pct))) > 1
);
 
-- 7. CHECK LOGICAL ERRORS
INSERT INTO dq_log (source_table, column_name, issue_description, row_affected, handling_rule, profiled_at)
 
-- customers -------------------------------------------------------------
SELECT
    'customers',
    'birth_date',
    'Birth date is in the future',
    COUNT(*),
    'REJECT: invalid future birth date',
    CURRENT_TIMESTAMP()
FROM customers
WHERE TRY_TO_DATE(TRIM(birth_date), 'yyyy-MM-dd') > CURRENT_DATE()
UNION ALL
SELECT
    'customers',
    'email',
    'Email does not match the expected basic email format',
    COUNT(*),
    'REVIEW: invalid email address',
    CURRENT_TIMESTAMP()
FROM customers
WHERE NULLIF(TRIM(email), '') IS NOT NULL AND TRIM(email) NOT RLIKE '^[^@\\s]+@[^@\\s]+\\.[^@\\s]{2,}$'
UNION ALL
SELECT
    'customers',
    'signup_date',
    'Signup date is before birth date',
    COUNT(*),
    'REVIEW: check source data',
    CURRENT_TIMESTAMP()
FROM customers
WHERE TRY_TO_DATE(TRIM(signup_date), 'yyyy-MM-dd') < TRY_TO_DATE(TRIM(birth_date), 'yyyy-MM-dd')
UNION ALL
 
-- customer_changes ------------------------------------------------------
SELECT
    'customer_changes',
    'customer_id',
    'Customer change references a nonexistent customer',
    COUNT(*),
    'REVIEW: orphan customer change',
    CURRENT_TIMESTAMP()
FROM customer_changes cc
WHERE cc.customer_id IS NOT NULL AND TRIM(cc.customer_id) <> '' AND NOT EXISTS (
    SELECT 1
    FROM customers c
    WHERE c.customer_id = cc.customer_id
)
UNION ALL
 
-- products --------------------------------------------------------------
SELECT
    'products',
    'unit_cost',
    'Unit cost is negative',
    COUNT(*),
    'REJECT: negative cost',
    CURRENT_TIMESTAMP()
FROM products
WHERE TRY_CAST(TRIM(unit_cost) AS DECIMAL(12,2)) < 0
UNION ALL
SELECT
    'products',
    'list_price',
    'List price is negative',
    COUNT(*),
    'REJECT: negative price',
    CURRENT_TIMESTAMP()
FROM products
WHERE TRY_CAST(TRIM(list_price) AS DECIMAL(12,2)) < 0
UNION ALL
SELECT
    'products',
    'list_price',
    'List price is lower than unit cost',
    COUNT(*),
    'REVIEW: list price below cost, business review',
    CURRENT_TIMESTAMP()
FROM products
WHERE TRY_CAST(TRIM(list_price) AS DECIMAL(12,2)) < TRY_CAST(TRIM(unit_cost) AS DECIMAL(12,2))
UNION ALL
SELECT
    'products',
    'is_active',
    'Active flag must be Y or N',
    COUNT(*),
    'REJECT: invalid active flag',
    CURRENT_TIMESTAMP()
FROM products
WHERE is_active IS NOT NULL AND TRIM(is_active) <> '' AND UPPER(TRIM(is_active)) NOT IN ('Y', 'N')
UNION ALL
 
-- stores ----------------------------------------------------------------
SELECT
    'stores',
    'open_date',
    'Store opening date is in the future',
    COUNT(*),
    'REVIEW: future opening date',
    CURRENT_TIMESTAMP()
FROM stores
WHERE TRY_TO_DATE(TRIM(open_date), 'yyyy-MM-dd') > CURRENT_DATE()
UNION ALL
SELECT
    'stores',
    'size_sqm',
    'Store size is negative',
    COUNT(*),
    'REJECT: negative store size',
    CURRENT_TIMESTAMP()
FROM stores
WHERE TRY_CAST(TRIM(size_sqm) AS INT) < 0
UNION ALL
 
-- orders ----------------------------------------------------------------
SELECT
    'orders',
    'order_ts',
    'Order timestamp is in the future',
    COUNT(*),
    'REVIEW: future-dated order',
    CURRENT_TIMESTAMP()
FROM orders
WHERE COALESCE(TRY_TO_TIMESTAMP(TRIM(order_ts), 'yyyy-MM-dd HH:mm:ss'), TRY_TO_TIMESTAMP(TRIM(order_ts), 'dd/MM/yyyy HH:mm')) > CURRENT_TIMESTAMP()
UNION ALL
SELECT
    'orders',
    'order_ts',
    'Order timestamp in dd/MM/yyyy HH:mm format instead of ISO',
    COUNT(*),
    'STANDARDISE: parse dd/MM/yyyy HH:mm with try_to_timestamp',
    CURRENT_TIMESTAMP()
FROM orders
WHERE TRY_TO_TIMESTAMP(TRIM(order_ts), 'yyyy-MM-dd HH:mm:ss') IS NULL
  AND TRY_TO_TIMESTAMP(TRIM(order_ts), 'dd/MM/yyyy HH:mm') IS NOT NULL
UNION ALL
SELECT
    'orders',
    'order_ts',
    'Order timestamp is earlier than the store opening date',
    COUNT(*),
    'REVIEW: order before store opened',
    CURRENT_TIMESTAMP()
FROM orders o
JOIN stores s ON o.store_id = s.store_id
WHERE COALESCE(TRY_TO_TIMESTAMP(TRIM(o.order_ts), 'yyyy-MM-dd HH:mm:ss'), TRY_TO_TIMESTAMP(TRIM(o.order_ts), 'dd/MM/yyyy HH:mm')) < CAST(TRY_TO_DATE(TRIM(s.open_date), 'yyyy-MM-dd') AS TIMESTAMP)
UNION ALL
SELECT
    'orders',
    'order_ts',
    'Order placed before customer signup date',
    COUNT(*),
    'REVIEW: check source data',
    CURRENT_TIMESTAMP()
FROM orders o
JOIN customers c ON o.customer_id = c.customer_id
WHERE CAST(COALESCE(TRY_TO_TIMESTAMP(TRIM(o.order_ts), 'yyyy-MM-dd HH:mm:ss'), TRY_TO_TIMESTAMP(TRIM(o.order_ts), 'dd/MM/yyyy HH:mm')) AS DATE) < TRY_TO_DATE(TRIM(c.signup_date),          'yyyy-MM-dd')
UNION ALL
SELECT
    'orders',
    'customer_id',
    'Order references a nonexistent customer',
    COUNT(*),
    'DEFAULT: map to unknown member (-1)',
    CURRENT_TIMESTAMP()
FROM orders o
WHERE o.customer_id IS NOT NULL AND TRIM(o.customer_id) <> '' AND NOT EXISTS (
    SELECT 1
    FROM customers c
    WHERE c.customer_id = o.customer_id
)
UNION ALL
SELECT
    'orders',
    'store_id',
    'Order references a nonexistent store',
    COUNT(*),
    'DEFAULT: map to unknown member (-1)',
    CURRENT_TIMESTAMP()
FROM orders o
WHERE o.store_id IS NOT NULL AND TRIM(o.store_id) <> '' AND NOT EXISTS (
    SELECT 1
    FROM stores s
    WHERE s.store_id = o.store_id
)
UNION ALL
SELECT
    'orders',
    'payment_method',
    'Payment method differs only by case or whitespace',
    COUNT(*),
    'STANDARDISE: trim and initcap',
    CURRENT_TIMESTAMP()
FROM orders
WHERE NULLIF(TRIM(payment_method), '') IS NOT NULL
  AND payment_method NOT IN ('Mobile Wallet', 'Card', 'Cash')
  AND INITCAP(TRIM(payment_method)) IN ('Mobile Wallet', 'Card', 'Cash')
UNION ALL
SELECT
    'orders',
    'payment_method',
    'Payment method not in permitted list',
    COUNT(*),
    'REJECT: unknown payment method',
    CURRENT_TIMESTAMP()
FROM orders
WHERE NULLIF(TRIM(payment_method), '') IS NOT NULL
  AND INITCAP(TRIM(payment_method)) NOT IN ('Mobile Wallet', 'Card', 'Cash')
UNION ALL
SELECT
    'orders',
    'order_status',
    'Order status differs only by case or whitespace',
    COUNT(*),
    'STANDARDISE: trim and initcap',
    CURRENT_TIMESTAMP()
FROM orders
WHERE NULLIF(TRIM(order_status), '') IS NOT NULL
  AND order_status NOT IN ('Cancelled', 'Completed', 'Returned')
  AND INITCAP(TRIM(order_status)) IN ('Cancelled', 'Completed', 'Returned')
UNION ALL
SELECT
    'orders',
    'order_status',
    'Order status not in permitted list',
    COUNT(*),
    'REJECT: unknown order status',
    CURRENT_TIMESTAMP()
FROM orders
WHERE NULLIF(TRIM(order_status), '') IS NOT NULL
  AND INITCAP(TRIM(order_status)) NOT IN ('Cancelled', 'Completed', 'Returned')
UNION ALL
 
-- order_items -----------------------------------------------------------
SELECT
    'order_items',
    'order_id',
    'Order item references a nonexistent order',
    COUNT(*),
    'REJECT: orphan order item (no order header)',
    CURRENT_TIMESTAMP()
FROM order_items oi
WHERE oi.order_id IS NOT NULL AND TRIM(oi.order_id) <> '' AND NOT EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.order_id = oi.order_id
)
UNION ALL
SELECT
    'order_items',
    'product_id',
    'Order item references a nonexistent product',
    COUNT(*),
    'REJECT: unknown product (no cost or margin)',
    CURRENT_TIMESTAMP()
FROM order_items oi
WHERE oi.product_id IS NOT NULL AND TRIM(oi.product_id) <> '' AND NOT EXISTS (
    SELECT 1
    FROM products p
    WHERE p.product_id = oi.product_id
)
UNION ALL
SELECT
    'order_items',
    'quantity',
    'Order item quantity is less than 1',
    COUNT(*),
    'REJECT: zero or negative quantity',
    CURRENT_TIMESTAMP()
FROM order_items
WHERE TRY_CAST(TRIM(quantity) AS INT) < 1
UNION ALL
SELECT
    'order_items',
    'unit_price',
    'Order item unit price is negative',
    COUNT(*),
    'REJECT: negative unit price',
    CURRENT_TIMESTAMP()
FROM order_items
WHERE TRY_CAST(TRIM(unit_price) AS DECIMAL(12,2)) < 0
UNION ALL
SELECT
    'order_items',
    'discount_pct',
    'Discount percentage is below 0',
    COUNT(*),
    'REJECT: negative discount',
    CURRENT_TIMESTAMP()
FROM order_items
WHERE TRY_CAST(TRIM(discount_pct) AS DOUBLE) < 0
UNION ALL
SELECT
    'order_items',
    'discount_pct',
    'Discount percentage above 1 and up to 100 (possible percent scale, e.g. 10 for 10%)',
    COUNT(*),
    'REVIEW: likely percent scale, candidate STANDARDISE (divide by 100)',
    CURRENT_TIMESTAMP()
FROM order_items
WHERE TRY_CAST(TRIM(discount_pct) AS DOUBLE) > 1
  AND TRY_CAST(TRIM(discount_pct) AS DOUBLE) <= 100
UNION ALL
SELECT
    'order_items',
    'discount_pct',
    'Discount percentage above 100',
    COUNT(*),
    'REJECT: discount out of range',
    CURRENT_TIMESTAMP()
FROM order_items
WHERE TRY_CAST(TRIM(discount_pct) AS DOUBLE) > 100;
 
-- 8. SUMMARY (put each of these two queries in its own cell)
SELECT UPPER(SPLIT_PART(handling_rule, ':', 1)) AS action,
       COUNT(*) AS checks,
       SUM(CASE WHEN row_affected > 0 THEN 1 ELSE 0 END) AS checks_with_findings,
       SUM(row_affected) AS sum_of_findings
FROM dq_log
GROUP BY 1
ORDER BY 1;
 
-- Only the checks that found something
SELECT source_table, column_name, issue_description, row_affected, handling_rule
FROM dq_log
WHERE row_affected > 0
ORDER BY source_table, column_name;

-- 7b. FINAL DECISIONS: every REVIEW becomes a real rule
UPDATE dq_log
SET handling_rule = CASE
    WHEN issue_description LIKE 'Invalid value: expected%' AND column_name = 'discount_pct'
        THEN 'REJECT: invalid discount value'
    WHEN issue_description LIKE 'Invalid value: expected%'
        THEN 'DEFAULT: set to NULL (an order date maps to unknown date -1)'
    WHEN issue_description LIKE 'Rows in a %group with conflicting data'
        THEN 'REJECT: all rows of the key rejected as CONFLICT_KEY'
    WHEN issue_description = 'Email does not match the expected basic email format'
        THEN 'DEFAULT: set to NULL'
    WHEN issue_description = 'Customer change references a nonexistent customer'
        THEN 'REJECT: orphan customer change (ORPHAN_FK)'
    WHEN issue_description = 'Store opening date is in the future'
        THEN 'DEFAULT: set to NULL'
    WHEN issue_description = 'Order timestamp is in the future'
        THEN 'DEFAULT: set to NULL, mapped to unknown date (-1)'
    WHEN issue_description LIKE 'Discount percentage above 1 and up to 100%'
        THEN 'STANDARDISE: divide by 100'
    WHEN issue_description IN ('Signup date is before birth date',
                               'List price is lower than unit cost',
                               'Order timestamp is earlier than the store opening date',
                               'Order placed before customer signup date')
        THEN 'FLAG: kept as received, business review'
    ELSE handling_rule
END
WHERE handling_rule LIKE 'REVIEW%';

-- Rules that said REJECT or "Unknown" but Silver actually defaults to NULL / Unknown
UPDATE dq_log SET handling_rule = 'DEFAULT: set to Unknown'
WHERE issue_description = 'Payment method not in permitted list';

UPDATE dq_log SET handling_rule = 'DEFAULT: set to NULL'
WHERE issue_description IN ('Active flag must be Y or N', 'Store size is negative');

UPDATE dq_log SET handling_rule = 'DEFAULT: set to NULL'
WHERE issue_description LIKE 'Missing or blank values found in column%'
  AND column_name IN ('email', 'birth_date', 'signup_date', 'open_date', 'size_sqm');

num_affected_rows
5
